# 101 · Engineering perspective mini lab

Companion to [Engineering perspective](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/101/engineer-perspective/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/101/engineering_perspective.ipynb)

**Goal:** walk the service decision sketch—JSON public API payload vs schemaless binary vs schema-driven sketch—and size the same logical DTO.

**Why this lab:** teams often pick a codec from habit or a single benchmark. Services need a short decision tree first: humans on the wire? shared IDL? who may produce the bytes?

**How to use:** run the decision function on scenarios, then compare encodings of one DTO, then a tiny validation sketch.

**Expect:** public REST → JSON family; multi-lang RPC → schema-driven; same-binary cache may allow native; JSON largest, binary sketches smaller; validation catches bad JSON even when parse succeeds.

> **Honesty banner:** sizes and timings here are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth.


In [ ]:
import json

# Public-ish API DTO
DTO = {
    "request_id": "req-7f3a",
    "user_id": 42,
    "items": [{"sku": "A-1", "qty": 2}, {"sku": "B-9", "qty": 1}],
    "metadata": {"source": "web", "attempt": 1},
}

try:
    import msgpack
    HAS_MSGPACK = True
except ImportError:
    HAS_MSGPACK = False
    print("optional: pip install msgpack")



## Decision sketch (from the article)

**Why:** the right first question is almost never “which library is fastest?”

**How:** classify each scenario with three flags: human-readable wire? need IDL/schema? single trusted runtime only?

**Expect:** four different answers for public REST, multi-lang RPC, private same-binary cache, and multi-service queues.

**Why it matters:** this sketch prevents shipping pickle or an undocumented binary map on a public boundary “because Results looked good.”


In [ ]:
def choose_family(human_readable: bool, need_idl: bool, single_trusted_runtime: bool) -> str:
    if human_readable:
        return "Text/JSON family (+ validation layer for real contracts)"
    if need_idl:
        return "Schema-driven (Protobuf/Avro-class)"
    if single_trusted_runtime:
        return "Language-native only inside a hard trust boundary"
    return "Schemaless binary (MessagePack/CBOR-class) + validation at edges"


scenarios = [
    ("Public REST body", True, False, False),
    ("Internal multi-lang RPC", False, True, False),
    ("Redis cache same service only", False, False, True),
    ("Internal queue, multi-service Python+Go", False, False, False),
]
for name, hum, idl, native in scenarios:
    print(f"{name:40} → {choose_family(hum, idl, native)}")



## Same DTO, three encodings

**Why:** density and debuggability trade off on the *same* logical payload.

**How:** encode one API-shaped dict as JSON, optional MessagePack, and a tiny field-number sketch.

**Expect:** JSON largest and readable; MessagePack smaller with keys still present; IDL-style sketch smaller still (names omitted)—teaching sketch, not production Protobuf.

**Why it matters:** size alone does not pick the format; public APIs often keep JSON for operability even when binary wins on bytes.


In [ ]:
def encode_varint(u: int) -> bytes:
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def encode_key(fn: int, wt: int) -> bytes:
    return encode_varint((fn << 3) | wt)


def encode_dto_idl(d: dict) -> bytes:
    # teaching sketch only: 1 request_id, 2 user_id, 3 items as JSON blob (lazy), 4 meta JSON
    out = bytearray()
    rid = d["request_id"].encode()
    out += encode_key(1, 2) + encode_varint(len(rid)) + rid
    out += encode_key(2, 0) + encode_varint(d["user_id"])
    items = json.dumps(d["items"], separators=(",", ":")).encode()
    out += encode_key(3, 2) + encode_varint(len(items)) + items
    meta = json.dumps(d["metadata"], separators=(",", ":")).encode()
    out += encode_key(4, 2) + encode_varint(len(meta)) + meta
    return bytes(out)


j = json.dumps(DTO, separators=(",", ":")).encode()
idl = encode_dto_idl(DTO)
print(f"{'encoding':22} {'nbytes':>6}")
print(f"{'JSON':22} {len(j):6}")
if HAS_MSGPACK:
    mp = msgpack.packb(DTO, use_bin_type=True)
    print(f"{'MessagePack':22} {len(mp):6}")
print(f"{'IDL sketch':22} {len(idl):6}")
print("JSON preview:", j.decode()[:80], "…")



## Validation still required for JSON

**Why:** “it parsed as JSON” ≠ “it is a valid request.”

**How:** check required keys and simple types on good vs bad payloads.

**Expect:** good DTO → no errors; partial/wrong-typed payload → explicit error list.

**Why it matters:** schemaless/text formats need an **external** contract (OpenAPI, JSON Schema, typed models). Schema-driven codecs move part of that into the IDL—not out of existence.


In [ ]:
REQUIRED = {"request_id", "user_id", "items"}


def validate_dto(d: dict) -> list:
    errs = []
    for k in REQUIRED:
        if k not in d:
            errs.append(f"missing {k}")
    if "user_id" in d and not isinstance(d["user_id"], int):
        errs.append("user_id must be int")
    if "items" in d and not isinstance(d["items"], list):
        errs.append("items must be list")
    return errs


good = json.loads(j)
bad = {"user_id": "42"}
assert validate_dto(good) == []
assert validate_dto(bad)
print("OK good:", validate_dto(good))
print("OK bad:", validate_dto(bad))



## Takeaways

1. Start from **human-readable?** → **IDL?** → **trust boundary?**
2. JSON wins public debuggability; binary wins density—under constraints.
3. Native formats are not an interchange strategy.

**Why it matters:** a repeatable decision path beats format fashion and misread microbenchmarks.

**Next:** [Data science lab](./data_science_perspective.ipynb) · [201 mechanisms](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/201/)
